<div class="lesson-banner">
<span class="lesson-kicker">Python course · 2-hour lesson</span>
<p>Measure bottlenecks and choose threads, processes, asyncio, batching, or vectorization from workload evidence.</p>
</div>

## Learning objectives

- Differentiate concurrency, parallelism, latency, and throughput.
- Choose threads for blocking I/O and processes for CPU-bound work.
- Use asyncio for many cooperative I/O tasks.
- Profile before optimizing and reason about memory growth.

::: {.callout-note}
### How to use this notebook
Read the explanation, predict each result, run the code, change the inputs, and complete the practice before revealing the solution.
:::


## Workload first

I/O-bound work spends time waiting for networks, disks, or databases; CPU-bound work spends time computing. Threads can overlap blocking I/O despite the GIL. Processes provide separate interpreters for CPU parallelism at the cost of serialization and startup. Asyncio excels when libraries expose non-blocking operations and task counts are high.


In [ ]:
from concurrent.futures import ThreadPoolExecutor
from time import sleep


def simulated_request(item_id: int) -> dict:
    sleep(0.02)
    return {"id": item_id, "status": "ok"}


with ThreadPoolExecutor(max_workers=4) as pool:
    results = list(pool.map(simulated_request, range(8)))
print(results)


## Asyncio is cooperative

An async task yields control at `await`, allowing the event loop to run other ready tasks. Blocking functions inside async code freeze the loop unless moved to a thread. Use bounded concurrency—unlimited tasks can overwhelm the client, server, sockets, or memory.


In [ ]:
import asyncio


async def simulated_fetch(item_id: int, semaphore: asyncio.Semaphore):
    async with semaphore:
        await asyncio.sleep(0.02)
        return {"id": item_id, "status": "ok"}


async def run_batch():
    semaphore = asyncio.Semaphore(4)
    tasks = [simulated_fetch(item_id, semaphore) for item_id in range(8)]
    return await asyncio.gather(*tasks)


# In a regular Python script: asyncio.run(run_batch())
# In a notebook with a running event loop: await run_batch()


## Measure time and memory

Optimize the dominant measured cost. Improve algorithms and data structures before micro-optimizing syntax. Stream data rather than materializing it, batch external operations, vectorize numerical work, cache only stable expensive results, and measure realistic inputs with repeated trials.


In [ ]:
from timeit import repeat

values = list(range(10_000))


def loop_squares():
    result = []
    for value in values:
        result.append(value * value)
    return result


def comprehension_squares():
    return [value * value for value in values]


for function in [loop_squares, comprehension_squares]:
    timings = repeat(function, number=100, repeat=5)
    print(function.__name__, min(timings))


## Worked example: bounded parallel I/O with result accounting

Every submitted item receives either a result or a captured error, and worker count stays bounded.


In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed


def transform(item: int) -> dict:
    if item < 0:
        raise ValueError("item cannot be negative")
    return {"input": item, "output": item * item}


items = [2, 4, -1, 8]
results, errors = [], []
with ThreadPoolExecutor(max_workers=3) as pool:
    futures = {pool.submit(transform, item): item for item in items}
    for future in as_completed(futures):
        item = futures[future]
        try:
            results.append(future.result())
        except ValueError as error:
            errors.append({"item": item, "error": str(error)})

print({"results": results, "errors": errors})


## Practice lab

Complete these tasks without copying the solution. Test normal, boundary, and invalid inputs where relevant.

1. Classify file download, image resize, database query, and matrix multiplication as I/O- or CPU-bound.
2. Process ten simulated I/O tasks with at most three workers.
3. Capture per-task errors without losing successful results.
4. Benchmark two correct implementations and explain whether the difference matters operationally.

::: {.callout-important}
### Practice standard
Your answer should be readable, deterministic, and divided into small functions when the task contains more than one rule.
:::


## Suggested solution

Open the folded code only after attempting every task.


In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from time import sleep


def io_task(item: int) -> int:
    sleep(0.01)
    if item == 6:
        raise RuntimeError("simulated remote failure")
    return item * 10


successes, failures = {}, {}
with ThreadPoolExecutor(max_workers=3) as pool:
    futures = {pool.submit(io_task, item): item for item in range(10)}
    for future in as_completed(futures):
        item = futures[future]
        try:
            successes[item] = future.result()
        except RuntimeError as error:
            failures[item] = str(error)

print({"successes": successes, "failures": failures})


## Knowledge check

**1. What is the GIL relevant to?**

::: {.callout-note collapse="true"}
### Answer
Execution of Python bytecode in threads within one CPython process.
:::

**2. Why bound concurrency?**

::: {.callout-note collapse="true"}
### Answer
To protect local and remote resources and control memory.
:::

**3. What should happen before optimization?**

::: {.callout-note collapse="true"}
### Answer
Measure a representative workload and identify the dominant cost.
:::


## Recap

- Match the model to the workload.
- Bound and account for concurrent work.
- Improve algorithms and I/O patterns before syntax-level tuning.


<div class="lesson-nav">
<a href="16-sql-and-databases.html"><i class="bi bi-arrow-left" aria-hidden="true"></i> SQL and Databases with Python</a>
<a href="18-python-for-ml-ai.html">Python for Machine Learning and AI Workflows <i class="bi bi-arrow-right" aria-hidden="true"></i></a>
</div>
